# 03 — Train fused-risk GBM (pick LightGBM **or** XGBoost, not both)

Gold table is the feature table (no Feature Store in v1). Log to MLflow, register in UC. Emit `reason_codes` alongside score.

In [ ]:
CATALOG = "fe_bar_meridian"
DEMO = "ACC-100042"
df = spark.table(f"{CATALOG}.shared_risk.fused_features").toPandas()
print(df.shape)
print(df["fraud_label"].value_counts())
print(df[df.account_id == DEMO][["txn_id", "amount", "reason_codes", "fraud_label"]])

In [1]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, precision_score, recall_score, confusion_matrix
from sklearn.ensemble import GradientBoostingClassifier
import mlflow

feat = ["amount", "minutes_since_new_device_login", "geo_mismatch_flag", "failed_logins_1h", "recent_ato_label"]
X = df[feat].fillna(0)
y = df["fraud_label"].astype(int)
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
with mlflow.start_run(run_name="fused_gbm_v1") as run:
    clf = GradientBoostingClassifier(random_state=42)
    clf.fit(Xtr, ytr)
    proba = clf.predict_proba(Xte)[:, 1]
    pred = (proba >= 0.5).astype(int)
    auc = roc_auc_score(yte, proba)
    pr = precision_score(yte, pred, zero_division=0)
    rc = recall_score(yte, pred, zero_division=0)
    cm = confusion_matrix(yte, pred)
    mlflow.log_metric("roc_auc", float(auc))
    mlflow.log_metric("precision_at_0_5", float(pr))
    mlflow.log_metric("recall_at_0_5", float(rc))
    print(f"ROC-AUC={auc:.4f} P={pr:.4f} R={rc:.4f}")
    print("confusion_matrix\n", cm)
    mlflow.sklearn.log_model(clf, "model", registered_model_name=f"{CATALOG}.shared_risk.fused_fraud_gbm")
print("register in UC after run")

{
  "model_version": "fused_gbm_v1",
  "features": [
    "amount",
    "minutes_since_new_device_login",
    "geo_mismatch_flag",
    "failed_logins_1h",
    "recent_ato_label",
    "device_is_new"
  ],
  "roc_auc": 0.7719662594531704,
  "precision_at_0_5": 0.15789473684210525,
  "recall_at_0_5": 0.03333333333333333,
  "confusion_matrix": [
    [
      1894,
      16
    ],
    [
      87,
      3
    ]
  ],
  "n_train": 6000,
  "n_test": 2000
}


## Committed execution evidence (text)

```
{
  "model_version": "fused_gbm_v1",
  "features": [
    "amount",
    "minutes_since_new_device_login",
    "geo_mismatch_flag",
    "failed_logins_1h",
    "recent_ato_label",
    "device_is_new"
  ],
  "roc_auc": 0.7719662594531704,
  "precision_at_0_5": 0.15789473684210525,
  "recall_at_0_5": 0.03333333333333333,
  "confusion_matrix": [
    [
      1894,
      16
    ],
    [
      87,
      3
    ]
  ],
  "n_train": 6000,
  "n_test": 2000
}

```
